# Pre-experiment: new-dataset feasibility only
Run separately for ttm and moirai1 on A100/Python 3.11 or 3.12. No main experiment, no test, no repeated old stability pilot. Four datasets: Solar, Traffic, Weather, Tetouan. Drive authorization is performed by the user. Installation and GPU results are pending until actually run.

In [ ]:
import json
import os
import subprocess
import sys
import zipfile
from pathlib import Path

assert (3, 11) <= sys.version_info[:2] <= (3, 12)
subprocess.run(["nvidia-smi"], check=True)
print("Kernel:", sys.executable, sys.version)
FAMILY = input("Model family (ttm or moirai1): ").strip()
assert FAMILY in ("ttm", "moirai1")
COMMIT = input("Full published pre-experiment preparation commit SHA: ").strip()
assert len(COMMIT) == 40 and all(c in "0123456789abcdef" for c in COMMIT)

In [ ]:
ROOT = Path("/content") / ("tsfm-pilot-" + COMMIT)
URL = "https://github.com/Han-Youseung/tsfm-zero-few-shot-crossover.git"
if not ROOT.exists():
    subprocess.run(["git", "clone", URL, str(ROOT)], check=True)
assert (
    subprocess.check_output(
        ["git", "-C", str(ROOT), "remote", "get-url", "origin"], text=True
    ).strip()
    == URL
)
assert not subprocess.check_output(
    ["git", "-C", str(ROOT), "status", "--porcelain", "--untracked-files=no"], text=True
).strip()
subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", COMMIT], check=True)
os.chdir(ROOT)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip() == COMMIT

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
PERSIST = Path("/content/drive/MyDrive/tsfm-pre-experiment")
OUT = PERSIST / COMMIT / FAMILY
OUT.mkdir(parents=True, exist_ok=True)
CACHE = PERSIST / "public-source-cache"
CACHE.mkdir(parents=True, exist_ok=True)
print("Persistent results and checkpoints:", OUT)
print("Stale running.lock: confirm old process is dead before manually removing only that lock.")

In [ ]:
ENV = Path("/content") / ("venv-pilot-" + FAMILY)
PY = ENV / "bin/python"
if not PY.exists():
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", f"python{version}-venv"], check=True)
    subprocess.run([sys.executable, "-m", "venv", str(ENV)], check=True)
subprocess.run(
    [str(PY), "-m", "pip", "install", "-r", f"requirements/{FAMILY}-gpu.txt"], check=True
)
subprocess.run([str(PY), "-m", "pip", "install", "-e", ".[dev]"], check=True)
subprocess.run([str(PY), "-m", "pip", "check"], check=True)
probe = (
    "import sys,torch; print(sys.executable,sys.version,torch.__version__,torch.version.cuda);"
    "assert torch.cuda.is_available(); print(torch.cuda.get_device_name(),"
    "torch.cuda.get_device_properties(0).total_memory)"
)
subprocess.run([str(PY), "-c", probe], check=True)
print("All model processes use:", PY, "; kernel imports no vendor package")

In [ ]:
# Raw cache persists on Drive; this checkout must be fresh for the cache link.
cache_link = ROOT / "data/source_cache"
if not cache_link.exists():
    cache_link.symlink_to(CACHE, target_is_directory=True)
subprocess.run([str(PY), "scripts/prepare_monash_variants.py"], check=True)
subprocess.run(
    [str(PY), "scripts/prepare_primary_candidates.py", "--cache", str(CACHE)], check=True
)
PINNED = ROOT / "results/manifests/pilot/prepared_primary.json"
entries = json.loads(PINNED.read_text())
NEW_DATASETS = {"Solar", "Traffic", "Weather", "Tetouan"}
assert all(entries[n]["status"] == "ready_with_warnings" for n in NEW_DATASETS)
print(
    {
        n: (entries[n]["qc"]["rows"], entries[n]["qc"]["channels"], entries[n]["qc"]["sha256"])
        for n in sorted(NEW_DATASETS)
    }
)

In [ ]:
BASE = [
    str(PY),
    "-m",
    "tsfm_crossover.experiments.pilot",
    "--prepared",
    str(PINNED),
    "--output",
    str(OUT),
    "--expected-commit",
    COMMIT,
]
subprocess.run(BASE, check=True)
plan = json.loads((OUT / "plan.json").read_text())
jobs = [
    r
    for r in plan["conditions"]
    if r["family"] == FAMILY
    and r["dataset"] in NEW_DATASETS
    and r["kind"] == "feasibility"
    and r["status"] == "planned"
]
assert len(jobs) == 16
jobs.sort(key=lambda r: (r["dataset"] != "Traffic", -r["horizon"], r["dataset"]))
for row in jobs:
    print(row["id"], row["dataset"], row["horizon"])
print("Exactly 16 new feasibility groups; no stability run; no test.")

In [ ]:
assert input("Type RUN_NEW to execute only the 16 groups above: ") == "RUN_NEW"
for row in jobs:
    done = subprocess.run(BASE + ["--condition-id", row["id"]])
    result_path = OUT / row["id"] / "result.json"
    result = json.loads(result_path.read_text()) if result_path.exists() else {}
    print(
        row["dataset"],
        row["horizon"],
        done.returncode,
        result.get("fp32_batch1_supported", "not_completed"),
    )
    if done.returncode != 0:
        print("Stopped on failed condition; preserve failure JSON and inspect before resuming.")
        break
print("Completed conditions are skipped on rerun. Larger-batch OOM is retained in attempts.")

In [ ]:
from google.colab import files

archive = Path("/content") / f"{FAMILY}-new-data-feasibility.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file() and p.suffix in {".json", ".csv"} and p.stat().st_size < 2_000_000:
            z.write(p, p.relative_to(OUT))
files.download(str(archive))
print("Drive is persistent. Export contains no weights/checkpoints/full predictions.")
print(
    "Return this archive for identity, data hash, model revision, eligibility and GPU checks; "
    "success is not automatic."
)